Imports

In [11]:
# ============================================================
# Window-to-Wall Ratio Segmentation Framework
# Intelligent Computing Version
# ============================================================

import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

Environment Configuration

In [12]:
# ============================================================
# Environment Configuration
# ============================================================

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

GPU Configuration

In [13]:
# ============================================================
# GPU Configuration
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))

TensorFlow : 2.20.0
GPU : []


Mixed Precision

In [14]:
# ============================================================
# Mixed Precision
# ============================================================

from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

print("Mixed Precision :", mixed_precision.global_policy())

Mixed Precision : <DTypePolicy "mixed_float16">


Mount Google Drive

In [15]:
# ============================================================
# Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Copy Dataset to Local SSD

In [16]:
# ============================================================
# Copy Dataset
# ============================================================

!cp "/content/drive/MyDrive/WWR_Seg_Model/data.zip" /content/

!unzip -oq /content/data.zip -d /content/

Configuration

In [17]:
@dataclass
class Config:

    IMAGE_SIZE = 256

    IMAGE_CHANNELS = 3

    MASK_CHANNELS = 1

    NUM_CLASSES = 4

    BATCH_SIZE = 16

    EPOCHS = 100

    LEARNING_RATE = 1e-4

    WEIGHT_DECAY = 1e-5

    BUFFER_SIZE = 1000

    SEED = 42

    TRAIN_RATIO = 0.90

    VAL_RATIO = 0.10

Segmentation Classes

In [18]:
# ============================================================
# Segmentation Classes
# ============================================================

CLASS_INFO = {
    0: {"name": "Roof",   "gray": 72},
    1: {"name": "Window", "gray": 128},
    2: {"name": "Wall",   "gray": 220},
    3: {"name": "Other",  "gray": 255},
}

CLASS_NAMES = [v["name"] for v in CLASS_INFO.values()]

GRAY_VALUES = [v["gray"] for v in CLASS_INFO.values()]

GRAY_TO_CLASS = {
    72: 0,
    128: 1,
    220: 2,
    255: 3
}

CLASS_TO_GRAY = {
    0: 72,
    1: 128,
    2: 220,
    3: 255
}

print("="*50)

print("Segmentation Classes")

print("="*50)

for idx in CLASS_INFO:

    print(
        f"{idx} -> "
        f"{CLASS_INFO[idx]['name']} "
        f"(Gray={CLASS_INFO[idx]['gray']})"
    )

Segmentation Classes
0 -> Roof (Gray=70)
1 -> Window (Gray=129)
2 -> Wall (Gray=221)
3 -> Other (Gray=255)


Project Paths

In [19]:
# ============================================================
# Project Paths
# ============================================================

# ---------- Local SSD ----------

LOCAL_ROOT = Path("/content")

IMAGE_DIR = LOCAL_ROOT / "images"

MASK_DIR = LOCAL_ROOT / "masks"

# ---------- Google Drive ----------

DRIVE_ROOT = Path("/content/drive/MyDrive/WWR_Seg_Model")

MODEL_DIR = DRIVE_ROOT / "models"

CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"

LOG_DIR = DRIVE_ROOT / "logs"

RESULT_DIR = DRIVE_ROOT / "results"

for folder in [

    MODEL_DIR,

    CHECKPOINT_DIR,

    LOG_DIR,

    RESULT_DIR

]:

    folder.mkdir(parents=True, exist_ok=True)

print("Local Dataset :", LOCAL_ROOT)

print("Project Root :", DRIVE_ROOT)

Local Dataset : /content
Project Root : /content/drive/MyDrive/WWR_Seg_Model


Detect Grayscale Values

In [24]:
# ============================================================
# Detect Grayscale Values
# ============================================================

all_values = set()

for mask_path in mask_files:

    mask = tf.io.decode_png(
        tf.io.read_file(str(mask_path)),
        channels=1
    )

    values = np.unique(mask.numpy())

    all_values.update(values.tolist())

print("Detected Gray Values:")
print(sorted(all_values))

Detected Gray Values:
[72, 128, 220, 255]


Verify Dataset

In [23]:
# ============================================================
# Verify Dataset
# ============================================================

print("=" * 60)
print("Verifying Dataset...")
print("=" * 60)

# ------------------------------------------------------------
# Check folders
# ------------------------------------------------------------

assert IMAGE_DIR.exists(), f"Image folder not found:\n{IMAGE_DIR}"
assert MASK_DIR.exists(), f"Mask folder not found:\n{MASK_DIR}"

# ------------------------------------------------------------
# Read file names
# ------------------------------------------------------------

image_files = sorted(IMAGE_DIR.glob("*.png"))
mask_files  = sorted(MASK_DIR.glob("*.png"))

assert len(image_files) > 0, "No images found."
assert len(mask_files) > 0, "No masks found."

assert len(image_files) == len(mask_files), \
    "Number of images and masks are different."

# ------------------------------------------------------------
# Check file names
# ------------------------------------------------------------

for img_path, mask_path in zip(image_files, mask_files):

    image_id = img_path.stem.replace("_texture", "")
    mask_id  = mask_path.stem.replace("_mask", "")

    assert image_id == mask_id, (
        f"Image/Mask mismatch:\n"
        f"{img_path.name}\n"
        f"{mask_path.name}"
    )

# ------------------------------------------------------------
# Check one sample image
# ------------------------------------------------------------

sample_image = tf.io.decode_png(
    tf.io.read_file(str(image_files[0])),
    channels=Config.IMAGE_CHANNELS
)

sample_mask = tf.io.decode_png(
    tf.io.read_file(str(mask_files[0])),
    channels=Config.MASK_CHANNELS
)

assert sample_image.shape[-1] == 3, \
    "Images must be RGB."

assert sample_mask.shape[-1] == 1, \
    "Masks must be Grayscale."

# ------------------------------------------------------------
# Check grayscale labels
# ------------------------------------------------------------

valid_values = {
    info["gray"]
    for info in CLASS_INFO.values()
}

for mask_path in mask_files:

    mask = tf.io.decode_png(
        tf.io.read_file(str(mask_path)),
        channels=1
    )

    values = np.unique(mask.numpy())

    invalid = set(values) - valid_values

    assert len(invalid) == 0, \
        f"Invalid grayscale values in {mask_path.name}: {invalid}"

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"Images            : {len(image_files)}")
print(f"Masks             : {len(mask_files)}")
print(f"Image Channels    : {sample_image.shape[-1]}")
print(f"Mask Channels     : {sample_mask.shape[-1]}")
print(f"Gray Values       : {GRAY_VALUES}")

print("\nDataset verification completed successfully.")

print("=" * 60)

Verifying Dataset...


AssertionError: Invalid grayscale values in 45.3677_-74.0326_mask.png: {np.uint8(72), np.uint8(220), np.uint8(128)}

Project Summary

In [ ]:
# ============================================================
# Project Summary
# ============================================================

print("="*60)

print("Window Segmentation Framework")

print("="*60)

print("Image Size :", Config.IMAGE_SIZE)

print("Batch Size :", Config.BATCH_SIZE)

print("Epochs     :", Config.EPOCHS)

print("Classes    :", Config.NUM_CLASSES)

print("Dataset    :", len(image_files))

print("="*60)